In [2]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'  # FORCE CPU

import tensorflow as tf
import random
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print("=" * 50)
print("Using CPU (GPU disabled for stability)")
print("=" * 50)
print(f"TensorFlow: {tf.__version__}")
print("✓ Ready to train\n")

Using CPU (GPU disabled for stability)
TensorFlow: 2.12.0
✓ Ready to train



## Define Dataset Path & Load Images

In [3]:
DATASET_PATH = "/mnt/d/waste-classification-system-II/dataset/New Trash Classfication Dataset/new-dataset-trash-type-v2"

image_paths = []
labels = []

class_names = sorted([
    cls for cls in os.listdir(DATASET_PATH)
    if os.path.isdir(os.path.join(DATASET_PATH, cls))
])

for label, cls in enumerate(class_names):
    class_path = os.path.join(DATASET_PATH, cls)
    
    for img_name in os.listdir(class_path):
        if img_name.lower().endswith((".jpg", ".jpeg", ".png")):
            image_paths.append(os.path.join(class_path, img_name))
            labels.append(label)

print("Classes:", class_names)
print("Total classes:", len(class_names))
print("Total images:", len(image_paths))

print("\nClass distribution:")
for label, cls in enumerate(class_names):
    print(f"{cls}: {labels.count(label)}")

Classes: ['cardboard', 'e-waste', 'glass', 'metal', 'organic', 'paper', 'plastic', 'textile', 'trash']
Total classes: 9
Total images: 8407

Class distribution:
cardboard: 893
e-waste: 993
glass: 948
metal: 901
organic: 967
paper: 853
plastic: 891
textile: 985
trash: 976


## Verify Sample Image|

In [4]:
idx = 0
img = tf.io.read_file(image_paths[idx])
img = tf.image.decode_image(img, channels=3)

print("Sample path:", image_paths[idx])
print("Image shape:", img.shape)
print("Label:", labels[idx], "->", class_names[labels[idx]])

Sample path: /mnt/d/waste-classification-system-II/dataset/New Trash Classfication Dataset/new-dataset-trash-type-v2/cardboard/cardboard1.jpg
Image shape: (384, 512, 3)
Label: 0 -> cardboard


## Train/Validation/Test Split

In [5]:
train_paths, temp_paths, y_train, y_temp = train_test_split(
    image_paths, labels, test_size=0.20, random_state=42, stratify=labels
)

val_paths, test_paths, y_val, y_test = train_test_split(
    temp_paths, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("Train:", len(train_paths))
print("Validation:", len(val_paths))
print("Test:", len(test_paths))

print("\nTrain distribution:", Counter(y_train))
print("Validation distribution:", Counter(y_val))
print("Test distribution:", Counter(y_test))

Train: 6725
Validation: 841
Test: 841

Train distribution: Counter({1: 794, 7: 788, 8: 781, 4: 774, 2: 758, 3: 721, 0: 714, 6: 713, 5: 682})
Validation distribution: Counter({1: 100, 7: 98, 8: 98, 4: 96, 2: 95, 0: 90, 3: 90, 6: 89, 5: 85})
Test distribution: Counter({1: 99, 7: 99, 4: 97, 8: 97, 2: 95, 3: 90, 0: 89, 6: 89, 5: 86})


## Create Dataset Pipeline

In [10]:
IMG_SIZE = 224  # MobileNetV2 standard (from 96)
BATCH_SIZE = 4   # Can use larger batch with MobileNet

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

train_ds = tf.data.Dataset.from_tensor_slices((train_paths, y_train))
train_ds = train_ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((val_paths, y_val))
val_ds = val_ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((test_paths, y_test))
test_ds = test_ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print(f"✓ Dataset ready")
print(f"  Image Size: {IMG_SIZE}x{IMG_SIZE}")
print(f"  Batch Size: {BATCH_SIZE}")

✓ Dataset ready
  Image Size: 224x224
  Batch Size: 4


## Build Model

In [11]:
num_classes = len(class_names)

# Use MobileNetV2 - optimized for edge devices
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),  # MobileNet standard
    include_top=False,
    weights='imagenet'
)

# Freeze base model weights (transfer learning)
base_model.trainable = False

# Build custom top layers
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(224, 224, 3)),
    
    # Preprocessing for MobileNetV2
    tf.keras.layers.Lambda(lambda x: tf.keras.applications.mobilenet_v2.preprocess_input(x)),
    
    # Pre-trained base
    base_model,
    
    # Custom classification head (lightweight)
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(f"Model parameters: {model.count_params():,}")
model.summary()

9406464/9406464 [==============================] - 3s 0us/step
Model parameters: 2,619,977
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lambda (Lambda)             (None, 224, 224, 3)       0         
                                                                 
 mobilenetv2_1.00_224 (Funct  (None, 7, 7, 1280)       2257984   
 ional)                                                          
                                                                 
 global_average_pooling2d (G  (None, 1280)             0         
 lobalAveragePooling2D)                                          
                                                                 
 dense (Dense)               (None, 256)               327936    
                                                                 
 dropout (Dropout)           (None, 256)               0         
                               

## Train Model 

In [12]:
import gc

gc.collect()
tf.keras.backend.clear_session()

# Phase 1: Train head only (frozen base)
print("Phase 1: Training classification head...")
EPOCHS_PHASE1 = 10

history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE1,
    verbose=1
)

# Phase 2: Fine-tune with lower learning rate
print("\nPhase 2: Fine-tuning base model...")
base_model = model.layers[2]
base_model.trainable = True

# Freeze early layers (keep learned features)
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),  # Lower LR
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

EPOCHS_PHASE2 = 5

history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE2,
    verbose=1
)

print("\n✓ Training completed!")

Phase 1: Training classification head...
Epoch 1/10
1682/1682 [==============================] - 127s 74ms/step - loss: 2.2109 - accuracy: 0.1152 - val_loss: 2.1962 - val_accuracy: 0.1189
Epoch 2/10
1682/1682 [==============================] - 124s 73ms/step - loss: 2.1968 - accuracy: 0.1124 - val_loss: 2.1961 - val_accuracy: 0.1189
Epoch 3/10
1682/1682 [==============================] - 129s 76ms/step - loss: 2.1968 - accuracy: 0.1161 - val_loss: 2.1960 - val_accuracy: 0.1189
Epoch 4/10
1682/1682 [==============================] - 120s 71ms/step - loss: 2.1968 - accuracy: 0.1081 - val_loss: 2.1960 - val_accuracy: 0.1189
Epoch 5/10
1682/1682 [==============================] - 124s 74ms/step - loss: 2.1966 - accuracy: 0.1146 - val_loss: 2.1960 - val_accuracy: 0.1189
Epoch 6/10
1682/1682 [==============================] - 121s 72ms/step - loss: 2.1966 - accuracy: 0.1135 - val_loss: 2.1960 - val_accuracy: 0.1189
Epoch 7/10
1682/1682 [==============================] - 121s 72ms/step - loss

AttributeError: 'GlobalAveragePooling2D' object has no attribute 'layers'

## Evaluate on Test Set

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)
print(f"\nTest Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")